# ICP For Point-Cloud Registration

Implementing the ICP algorithm to register two point clouds. 
- Registration refers to aligning to be coherent. 
- In robotics, when lidar scans from different parts of a room, for instance, need to be joined together to create a full map of the environment.
- Some examples in [this paper](http://redwood-data.org/indoor_lidar_rgbd/paper.pdf).

## Setup 

- This notebook originally from [CS237a hw3, part3](https://colab.research.google.com/drive/1O94OB54oYSu7E1CAxP29IeuCnfJnA3t8?usp=sharing)
- Use the [Stanford Bunny](https://graphics.stanford.edu/data/3Dscanrep/) dataset
- Referring to partial point cloud, $A$, as the "source", 
- Referring to full scan, $B$, of the bunny as the "target"

```python
cloud_A = o3d.io.read_point_cloud("data/bun045.ply")
cloud_B = o3d.io.read_point_cloud("data/bun_zipper.ply")
```

In [ ]:
# In terminal, create a virtual env, and install dependencies
#
# uv venv
# source .venv/bin/activate
# uv pip install open3d 
# uv pip install ipykernel
# uv pip install ipywidgets
# code .
#  
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from utils import *

cloud_A = o3d.io.read_point_cloud("data/bun045.ply")
cloud_B = o3d.io.read_point_cloud("data/bun_zipper.ply")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Visualize point cloud data

- Use the slider bars to view the bunny from different angles.
- Partial point cloud $A$ scan is not a complete scan of the bunny
- We will be referring to this partial point cloud, $A$, as the "source", 
- We will be referring to the full scan $B_t$ of the bunny as the "target"

In [3]:
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_A]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

In [4]:
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

In [5]:
# View both clouds A (partial_scan) and B (full_bunny)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_A, cloud_B]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

# Global Registration

Our goal will be to match the point cloud data
- containing a partial view of the bunny (A)
- with the full 3D mesh model of the same bunny (B)

We will do this using the RANSAC algorithm.


$$ 
\text{Given A, B} \\
\text{find a matrix} \quad T \\
\text{s.t.} \quad B = A T 
$$

## Extract features and downsample
We can see that the two point clouds are clearly misaligned right now. 

- So first, we will extract some features of each point cloud. 
- These features, called the "FPFH" features of the point clouds are a vector of 33 values for each point in the point cloud that represents some unique features of that point. 
- Therefore, if we have N points in point cloud, your FPFH feature matrix for that point cloud will be of shape (N, 33).

Since N can be large for raw point cloud data, we will downsample it a bit so that it is easier to experiment with.

In [6]:
# Extract features
def preprocess_point_cloud(pcd, voxel_size):
    print(":: Downsample with a voxel size %.3f." % voxel_size)
    pcd_down = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    print(":: Estimate normal with search radius %.3f." % radius_normal)
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
    return pcd_down, pcd_fpfh

def prepare_dataset(A, B, voxel_size):
    A_sample, A_fpfh = preprocess_point_cloud(A, voxel_size)
    B_sample, B_fpfh = preprocess_point_cloud(B, voxel_size)
    return A, B, A_sample, B_sample, A_fpfh, B_fpfh

cloud_A, cloud_B, A_sample, B_sample, A, B = prepare_dataset(cloud_A, cloud_B, 0.01)

:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.
:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.


In [7]:
# RANSAC algorithm using open3d
def register(A_sample, B_sample, A, B, voxel_size):
    distance_threshold = voxel_size * 1.5
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        A_sample,
        B_sample, 
        A, 
        B, 
        False, 
        distance_threshold
    )
    
    T = result.transformation
    return T 

T = register(A_sample, B_sample, A, B, 0.01)
print(f"T.shape {T.shape} \n {T}")


T.shape (4, 4) 
 [[ 8.23834935e-01  1.73241033e-02  5.66564979e-01 -5.32572700e-02]
 [-4.29309974e-02  9.98568899e-01  3.18917431e-02  5.54942818e-04]
 [-5.65201671e-01 -5.05967317e-02  8.23399685e-01 -6.11093701e-03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


# After RANSAC
Let's now apply this transformation matrix to our point cloud data to see how well the point clouds are now registered.

In [8]:
from copy import deepcopy

A_transformed  = deepcopy(cloud_A).transform(T)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

As you can see, `A.transform(T)` and `B: full_bunny` are much better aligned than before but need fine-tuning.

We will now use ICP to try to improve this alignment, a process also called "Local Refinement"

# Basic ICP

In this section, we will implement a basic version of the ICP algorithm. The ICP algorithm has the following steps:



1.   For every point in the source point cloud, find its nearest neighbor in the target point cloud
2.   Find a matrix T that maps points in the source point cloud to the target point cloud while minimizing the euclidean distance between a point and its nearest neighbor (this will be our **error** for each point)
3. Apply this transformation, T, to the source point cloud data
4. Compute the average error for this transformation across all points in the source point cloud data
5. Repeat steps 1 - 4 for N iterations, or break if:

$abs(e_t - e_{t-1}) < \tau$

where $e_i$ is the average error in the i-th iteration and $\tau$ is the tolerance set as a hyperparameter.


In [9]:
from sklearn.neighbors import NearestNeighbors
from tqdm import trange 

def best_fit_transform(A, B):
    m = A.shape[1]

    # Compute cloud centroids
    Ac = A.mean(axis=0)
    Bc = B.mean(axis=0)

    # Center the points
    AA = A - Ac
    BB = B - Bc
    
    H = AA.T @ BB

    # SVD
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # special reflection case
    if np.linalg.det(R) < 0:
       Vt[-1] = -Vt[-1]
       R = Vt.T @ U.T

    # translation
    t = Bc.T - (R @ Ac.T)

    # homogeneous transformation
    T = np.identity(m+1)
    T[:m, :m] = R
    T[:m, m] = t

    return T, R, t

def nearest_neighbor(src, dst, radius=0.01):
    neigh = NearestNeighbors(n_neighbors=1)
    neigh.fit(dst)
    distances, indices = neigh.kneighbors(src, return_distance=True)
    return distances.ravel(), indices.ravel()

def icp(A, B, init_pose=None, max_iterations=20, tolerance=0.001, knn_radius=0.01):
    m = A.shape[1]

    # make points homogeneous, copy them to maintain the originals
    src = np.ones((m+1, A.shape[0]))
    dst = np.ones((m+1, B.shape[0]))

    src[:m,:] = np.copy(A.T)
    dst[:m,:] = np.copy(B.T)

    # apply the initial pose estimation
    if init_pose is not None:
        src = np.dot(init_pose, src)

    prev_error = 0
    for i in trange(max_iterations):

        distances, indices = nearest_neighbor(src[:m, :].T, dst[:m, :].T, radius=knn_radius)

        T_iter, _, _ = best_fit_transform(src[:m, :].T, dst[:m, indices].T)
        src = np.dot(T_iter, src)

        mean_error = np.mean(distances)
        if np.abs(mean_error - prev_error) < tolerance:
            print(f"converged iteration: {i}, rmse: {mean_error:.6e}")
            break
        prev_error = mean_error
    
    # calculate final transformation
    T, _, _ = best_fit_transform(A, src[:m,:].T)
    return T


A_np = np.asarray(A_sample.points)
B_np = np.asarray(B_sample.points)
T_icp = icp(A_np, B_np, T, max_iterations=20, tolerance=1e-6, knn_radius=1e-2)
T_transformed_ICP = deepcopy(cloud_A).transform(T_icp) # transformed complete size pcd

widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, T_transformed_ICP]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

 60%|██████    | 12/20 [00:00<00:00, 809.72it/s]

converged iteration: 12, rmse: 3.060709e-03


interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

# Robust ICP Using open3D

In [13]:
threshold = 0.01
max_iterations=3000

print("Apply point-to-point ICP")
point2point_registration = o3d.pipelines.registration.registration_icp(
    cloud_A,
    cloud_B,
    threshold,
    T,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=max_iterations)
)

T_robust = point2point_registration.transformation
print(f"{point2point_registration} \n Transformation: {T_robust}")

A_transformed_robust = deepcopy(cloud_A).transform(T_robust)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed_robust]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation"))

Apply point-to-point ICP
RegistrationResult with fitness=1.000000e+00, inlier_rmse=5.464823e-04, and correspondence_set size of 40097
Access transformation to get result. 
 Transformation: [[ 8.26009889e-01 -1.03092039e-02  5.63561340e-01 -5.19880316e-02]
 [ 2.42897980e-03  9.99888546e-01  1.47307720e-02 -2.06939874e-04]
 [-5.63650392e-01 -1.07988842e-02  8.25942867e-01 -1.07526675e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>